# **Nebula Core** — serve + tunnel (free T4) · v2

**Important:** every Colab notebook gets its OWN machine. The serve notebook
does not share files with the training notebook. So your trained GGUF must be
reachable from THIS machine — via this session's disk, an upload, or Google Drive.

Cell 1 handles all of that automatically and prints a full diagnostic if the
model is not found. If you hit any error anywhere: run the LAST cell
(diagnostics) and paste its output into the Nebula chat.

Order: locate model → persist to Drive → install GPU runtime (llama.cpp,
auto-fallback to Ollama) → serve OpenAI-compatible on the T4 → public HTTPS
tunnel → self-test one real Nebula build prompt.

In [ ]:
# ── 1. Find the trained GGUF (session disk → upload → Google Drive) ──
import glob, os, textwrap

print('GPU check:');
os.system('nvidia-smi -L || true')

def scan():
    hits = []
    for d in ['/content', '/content/nebula-core/gguf', '/content/drive/MyDrive/nebula-core']:
        hits += sorted(glob.glob(os.path.join(d, '*.gguf')))
    # prefer q4_k_m, then anything
    hits = [h for h in hits if 'q4_k_m' in h.lower()] + [h for h in hits if 'q4_k_m' not in h.lower()]
    return hits

GGUF = scan()[0] if scan() else None

if not GGUF:
    # last chance: mount Drive now (it may not be mounted yet) and rescan
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        GGUF = scan()[0] if scan() else None
    except Exception as e:
        print('Drive mount failed:', e)

if GGUF:
    print(f'\nFOUND MODEL: {GGUF}  ({os.path.getsize(GGUF)/1e9:.2f} GB)')
else:
    print('\n' + '='*62)
    print('NO GGUF FOUND ON THIS MACHINE. Here is exactly what exists:')
    print('='*62)
    print('/content/            ->', os.listdir('/content')[:20])
    try:
        print('/content/drive       ->', os.listdir('/content/drive/MyDrive')[:20])
    except Exception:
        print('/content/drive       -> not mounted')
    print(textwrap.dedent('''
    RECOVER with any ONE of these:
      A) Training tab still open and alive? Run this ONE cell THERE
         (it copies the model into Drive, ~2 min):
             from google.colab import drive; drive.mount('/content/drive')
             import shutil, glob, os
             os.makedirs('/content/drive/MyDrive/nebula-core', exist_ok=True)
             for f in glob.glob('/content/nebula-core/gguf/*.gguf'):
                 dst = '/content/drive/MyDrive/nebula-core/' + os.path.basename(f)
                 if not os.path.exists(dst): shutil.copy(f, dst); print('saved', dst)
         Then re-run THIS notebook.
      B) GGUF downloaded to your computer (cell 7 of training)?
         Upload it at drive.google.com into a folder named 'nebula-core',
         then re-run THIS notebook.
      C) Neither? Re-run the UPDATED training notebook (Run all, ~90 min,
         hands-free) — it now auto-saves the GGUF to Drive. Then re-run THIS.
    '''))
    raise RuntimeError('model not found — follow A, B or C above')

In [ ]:
# ── 2. Persist to Google Drive (skip if already there) ─────────
import glob, os, shutil
if not GGUF.startswith('/content/drive'):
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        dst_dir = '/content/drive/MyDrive/nebula-core'
        os.makedirs(dst_dir, exist_ok=True)
        for f in glob.glob(os.path.dirname(GGUF) + '/*.gguf'):
            dst = os.path.join(dst_dir, os.path.basename(f))
            if not os.path.exists(dst):
                shutil.copy(f, dst)
                print('saved ->', dst)
        print('Drive copy complete:', os.listdir(dst_dir))
    except Exception as e:
        print('WARNING: could not persist to Drive (', e, ') — serving from session disk.')
else:
    print('already on Drive — nothing to copy')

In [ ]:
# ── 3. GPU runtime: llama.cpp prebuilt → Ollama fallback → CPU last resort ──
import json, re, subprocess, os, time

def sh(cmd, timeout=600):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)

RUNTIME = None
PORT = None

# (a) prebuilt llama.cpp CUDA binary — scan the release page (no API, no rate limit)
try:
    tag = sh('curl -fsSLI -o /dev/null -w %{url_effective} '
            'https://github.com/ggml-org/llama.cpp/releases/latest', timeout=60).stdout.strip().rsplit('/', 1)[-1]
    html = sh(f'curl -fsSL https://github.com/ggml-org/llama.cpp/releases/expanded_assets/{tag}', timeout=60).stdout
    m = re.search(r'href="([^"]*ubuntu[^"]*cuda[^"]*x64\.zip)"', html)
    if m:
        url = 'https://github.com' + m.group(1).replace('&amp;', '&')
        print('downloading prebuilt CUDA llama.cpp:', url.rsplit('/', 1)[-1])
        r = sh(f'curl -fsSL -o /tmp/llama.zip "{url}" && unzip -o -q /tmp/llama.zip -d /tmp/llamacpp', timeout=300)
        exe = sh('ls /tmp/llamacpp/*/llama-server /tmp/llamacpp/llama-server 2>/dev/null | head -1').stdout.strip()
        if exe and os.path.exists(exe):
            sh(f'chmod +x "{exe}" && ln -sf "{exe}" /usr/local/bin/llama-server')
            RUNTIME, PORT = 'llama', 8080
            print('prebuilt llama-server ready (GPU)')
except Exception as e:
    print('prebuilt llama.cpp skipped:', str(e)[:200])

# (b) Ollama fallback — the most robust GPU server on Colab
if not RUNTIME:
    print('installing Ollama (auto-GPU, ~2 min)…')
    r = sh('curl -fsSL https://ollama.com/install.sh | sh', timeout=600)
    if r.returncode == 0:
        RUNTIME, PORT = 'ollama', 11434
        print('ollama installed:', sh('ollama --version', timeout=60).stdout.strip())
    else:
        print(r.stderr[-800:])

# (c) last resort: CPU-only wheel (works but SLOW — warning)
if not RUNTIME:
    print('falling back to CPU llama-cpp-python — this WILL be slow (~10x slower)')
    r = sh('pip install -q llama-cpp-python', timeout=600)
    if r.returncode != 0:
        print(r.stderr[-800:]); raise RuntimeError('no runtime could be installed')
    RUNTIME, PORT = 'pycpu', 8080

# cloudflared for the public tunnel
r = sh('curl -fsSL -o /usr/local/bin/cloudflared '
       'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && '
       'chmod +x /usr/local/bin/cloudflared', timeout=120)
if r.returncode != 0:
    print(r.stderr[-500:]); raise RuntimeError('cloudflared download failed')
print('runtime =', RUNTIME, '| port =', PORT)
print('cloudflared ready:', sh('cloudflared --version', timeout=60).stdout.strip())

In [ ]:
# ── 4. Start the model server (GPU) ────────────────────────────
import subprocess, time, requests, os

subprocess.run('pkill -f llama-server; pkill -f llama_cpp; pkill -f "ollama serve"',
               shell=True, capture_output=True)

if RUNTIME == 'llama':
    subprocess.run(f'nohup llama-server -m "{GGUF}" --port 8080 --host 0.0.0.0 '
                   f'--ctx-size 8192 --parallel 2 --alias nebula-core '
                   f'> /tmp/llama.log 2>&1 &', shell=True)
    HEALTH = 'http://127.0.0.1:8080/health'
elif RUNTIME == 'ollama':
    subprocess.run('nohup ollama serve > /tmp/ollama.log 2>&1 &', shell=True)
    for _ in range(30):
        try:
            if requests.get('http://127.0.0.1:11434/', timeout=2).status_code == 200:
                break
        except Exception:
            time.sleep(1)
    open('/tmp/Modelfile', 'w').write(f'FROM {GGUF}\nPARAMETER num_ctx 8192\nPARAMETER temperature 0.4\n')
    cr = subprocess.run('ollama create nebula-core -f /tmp/Modelfile', shell=True,
                        capture_output=True, text=True, timeout=900)
    print(cr.stdout[-400:], cr.stderr[-400:])
    HEALTH = 'http://127.0.0.1:11434/v1/models'
else:
    subprocess.run(f'nohup python -m llama_cpp.server --model "{GGUF}" '
                   f'--host 0.0.0.0 --port 8080 --n_ctx 8192 > /tmp/llama.log 2>&1 &', shell=True)
    HEALTH = 'http://127.0.0.1:8080/docs'

up = False
for i in range(120):
    try:
        if requests.get(HEALTH, timeout=3).status_code == 200:
            up = True
            print(f'model server healthy after ~{i*2}s')
            break
    except Exception:
        pass
    time.sleep(2)
if not up:
    logf = '/tmp/ollama.log' if RUNTIME == 'ollama' else '/tmp/llama.log'
    print('SERVER DID NOT COME UP — log tail of', logf)
    print(open(logf).read()[-1500:])
    raise RuntimeError(f'server not healthy ({RUNTIME}) — see log above')

In [ ]:
# ── 5. Public HTTPS tunnel ─────────────────────────────────────
import subprocess, re, time

subprocess.run('pkill -f cloudflared', shell=True, capture_output=True)
subprocess.run(f'nohup cloudflared tunnel --url http://localhost:{PORT} '
               '> /tmp/tunnel.log 2>&1 &', shell=True)

TUNNEL_URL = None
for i in range(60):
    time.sleep(2)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/tmp/tunnel.log').read())
    if m:
        TUNNEL_URL = m.group(0)
        break
if not TUNNEL_URL:
    print('tunnel log tail:'); print(open('/tmp/tunnel.log').read()[-1200:])
    raise RuntimeError('tunnel did not start')
print('PUBLIC URL :', TUNNEL_URL)
print()
print('LLM_CUSTOM_BASE_URL = ' + TUNNEL_URL + '/v1')
print('LLM_CUSTOM_MODEL    = nebula-core')
print()
print('>>> Paste the two lines above into the Nebula chat. <<<')
print('>>> Keep this Colab tab open while serving. <<<')

In [ ]:
# ── 6. Self-test: one REAL Nebula build prompt through YOUR model ─
import requests, time

msgs = [
    {"role": "system", "content": "You are Nebula Core, the fine-tuned engineering model of the Nebula platform. You build production-quality websites on the FIRST round. Output exactly the format asked."},
    {"role": "user", "content": "Section: cta. Requirement: a closing CTA band: one promise, one button, no clutter. Business: dental clinic group. Brand tokens: bg #faf7f2, ink #191919, accent #0e7c66. Output the complete <section> with scoped <style>."},
]
t0 = time.time()
r = requests.post(f'http://127.0.0.1:{PORT}/v1/chat/completions', json={
    'model': 'nebula-core', 'messages': msgs,
    'temperature': 0.4, 'max_tokens': 700,
}, timeout=600)
out = r.json()['choices'][0]['message']['content']
secs = time.time() - t0
print(f'generated {len(out)} chars in {secs:.1f}s  (~{len(out.split())/max(secs,0.1):.1f} tok/s)')
print('--- first 900 chars ---')
print(out[:900])
assert '<section' in out.lower(), 'model did not emit a section — check training'
print()
print('SELF-TEST PASSED — your own model, building, on your own weights.')

In [ ]:
# ── 7. DIAGNOSTICS — run this if ANYTHING above errored ────────
# Then paste the whole output into the Nebula chat.
import subprocess, os
def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=60)
    return (r.stdout or '') + (r.stderr or '')
print('== GPU ==');            print(run('nvidia-smi || echo no-gpu')[:800])
print('== disk ==');           print(run('df -h /content | tail -1'))
print('== files ==');          print('/content:', os.listdir('/content')[:15])
try:                           print('/content/nebula-core/gguf:', os.listdir('/content/nebula-core/gguf'))
except Exception as e:         print('gguf dir:', e)
for f in ['/tmp/llama.log', '/tmp/ollama.log', '/tmp/tunnel.log']:
    print(f'== {f} (tail) ==')
    try:    print(open(f).read()[-900:])
    except Exception as e: print('no log:', e)